# Enterprise SIEM Lab: Wazuh Cluster with Custom Alert Dashboards
**Author:** Jatin Jangid  |  **Program:** Cybersecurity Bootcamp  |  **Date:** 22-10-2026  |  **Project:** Wazuh SIEM Lab

Deployed a multi-node Wazuh cluster (manager master/worker, indexer, dashboard) to aggregate endpoint
and network logs, with custom detection rules and alert dashboards.

**Notebook contents**
1. Architecture and deployment commands
2. Custom detection rules (MITRE ATT&CK mapped)
3. Wazuh API: cluster and agent health
4. Alert analysis from the Wazuh indexer (falls back to sample data)
5. Visualizations / dashboard summary

## 1. Architecture and deployment
Reference commands run on the lab hosts (shown, not executed here).

In [ ]:
DEPLOY_STEPS = """
# --- Indexer + Dashboard + Manager (all-in-one or distributed) ---
curl -sO https://packages.wazuh.com/4.x/wazuh-install.sh
curl -sO https://packages.wazuh.com/4.x/config.yml     # define node names/IPs
sudo bash wazuh-install.sh --generate-config-files
sudo bash wazuh-install.sh --wazuh-indexer  node-1
sudo bash wazuh-install.sh --start-cluster
sudo bash wazuh-install.sh --wazuh-server   wazuh-1    # master
sudo bash wazuh-install.sh --wazuh-server   wazuh-2    # worker
sudo bash wazuh-install.sh --wazuh-dashboard dashboard

# --- Enroll an agent (Linux example) ---
WAZUH_MANAGER='<MANAGER_IP>' WAZUH_AGENT_GROUP='linux' apt install wazuh-agent
systemctl enable --now wazuh-agent

# --- Cluster check ---
/var/ossec/bin/cluster_control -l
"""
print(DEPLOY_STEPS)

## 2. Custom detection rules
Placed in `/var/ossec/etc/rules/local_rules.xml` on the manager.

In [ ]:
LOCAL_RULES = r"""
<group name="local,custom,">

  <!-- Repeated SSH failures from one source: brute force (T1110) -->
  <rule id="100100" level="10" frequency="6" timeframe="60">
    <if_matched_sid>5760</if_matched_sid>
    <same_source_ip />
    <description>Custom: SSH brute force from $(srcip)</description>
    <mitre><id>T1110</id></mitre>
  </rule>

  <!-- New local user created (T1136.001) -->
  <rule id="100110" level="8">
    <if_sid>5902</if_sid>
    <description>Custom: new local user account created</description>
    <mitre><id>T1136.001</id></mitre>
  </rule>

  <!-- Sysmon: PowerShell encoded command (T1059.001) -->
  <rule id="100120" level="12">
    <if_group>sysmon_event1</if_group>
    <field name="win.eventdata.commandLine" type="pcre2">(?i)powershell.*-enc(odedcommand)?\s</field>
    <description>Custom: PowerShell encoded command executed</description>
    <mitre><id>T1059.001</id></mitre>
  </rule>

</group>
"""
with open("local_rules.xml", "w") as f:
    f.write(LOCAL_RULES)
print("Wrote local_rules.xml")

## 3. Wazuh API: cluster and agent health
Set `LIVE = True` and fill credentials to query a real cluster.

In [ ]:
import requests, urllib3, json
urllib3.disable_warnings()

LIVE = False
API_URL  = "https://<MANAGER_IP>:55000"
API_USER = "wazuh-wui"
API_PASS = "CHANGE_ME"

def api_token():
    r = requests.post(f"{API_URL}/security/user/authenticate?raw=true",
                      auth=(API_USER, API_PASS), verify=False, timeout=10)
    r.raise_for_status()
    return r.text

def api_get(path, token, **params):
    r = requests.get(f"{API_URL}{path}", headers={"Authorization": f"Bearer {token}"},
                     params=params, verify=False, timeout=10)
    r.raise_for_status()
    return r.json()["data"]

if LIVE:
    t = api_token()
    print(json.dumps(api_get("/cluster/status", t), indent=2))
    nodes = api_get("/cluster/nodes", t)["affected_items"]
    agents = api_get("/agents", t, select="id,name,status,os.name", limit=500)["affected_items"]
    print(f"{len(nodes)} cluster nodes, {len(agents)} agents")
else:
    print("LIVE=False: skipping API calls")

## 4. Alert analysis
Queries the Wazuh indexer (`wazuh-alerts-*`). Without a cluster, synthetic alerts are generated so the notebook runs end to end.

In [ ]:
import pandas as pd, numpy as np

INDEXER_URL, IDX_USER, IDX_PASS = "https://<INDEXER_IP>:9200", "admin", "CHANGE_ME"

def fetch_alerts(hours=24, size=5000):
    q = {"size": size, "sort": [{"timestamp": "desc"}],
         "query": {"range": {"timestamp": {"gte": f"now-{hours}h"}}}}
    r = requests.post(f"{INDEXER_URL}/wazuh-alerts-*/_search", json=q,
                      auth=(IDX_USER, IDX_PASS), verify=False, timeout=30)
    r.raise_for_status()
    rows = []
    for h in r.json()["hits"]["hits"]:
        s = h["_source"]
        rows.append({"timestamp": s["timestamp"], "agent": s["agent"]["name"],
                     "rule_id": s["rule"]["id"], "level": s["rule"]["level"],
                     "description": s["rule"]["description"],
                     "mitre": ",".join(s["rule"].get("mitre", {}).get("id", []))})
    return pd.DataFrame(rows)

def sample_alerts(n=600, seed=7):
    rng = np.random.default_rng(seed)
    rules = [("5760","SSH authentication failed",5,"T1110"),
             ("100100","Custom: SSH brute force",10,"T1110"),
             ("100110","Custom: new local user created",8,"T1136.001"),
             ("100120","Custom: PowerShell encoded command",12,"T1059.001"),
             ("550","File integrity: file modified",7,"T1565.001"),
             ("60122","Windows logon failure",5,"T1078")]
    w = [.35,.1,.05,.05,.25,.2]
    agents = ["web-01","db-01","win-dc01","win-ws01","kali-target"]
    idx = rng.choice(len(rules), n, p=w)
    ts = pd.Timestamp.now() - pd.to_timedelta(rng.integers(0, 24*60, n), unit="m")
    return pd.DataFrame({"timestamp": ts, "agent": rng.choice(agents, n),
        "rule_id":[rules[i][0] for i in idx], "description":[rules[i][1] for i in idx],
        "level":[rules[i][2] for i in idx], "mitre":[rules[i][3] for i in idx]})

try:
    if not LIVE: raise RuntimeError("sample mode")
    df = fetch_alerts()
except Exception as e:
    print("Using sample data:", e)
    df = sample_alerts()

df["timestamp"] = pd.to_datetime(df["timestamp"])
print(len(df), "alerts")
df.head()

In [ ]:
summary = {
    "total_alerts": len(df),
    "high_severity (>=10)": int((df.level >= 10).sum()),
    "agents_reporting": df.agent.nunique(),
    "custom_rule_hits": int(df.rule_id.astype(str).str.startswith("1001").sum()),
}
print(json.dumps(summary, indent=2))
df.groupby(["rule_id","description"]).size().sort_values(ascending=False).head(10)

## 5. Dashboard-style visualizations

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(2, 2, figsize=(13, 8))

df.groupby("description").size().sort_values().plot.barh(ax=ax[0,0], title="Alerts by rule")
df.groupby("agent").size().sort_values().plot.barh(ax=ax[0,1], title="Alerts by agent")
df.set_index("timestamp").resample("1h").size().plot(ax=ax[1,0], title="Alert volume per hour")
df[df.mitre != ""].groupby("mitre").size().sort_values().plot.barh(ax=ax[1,1], title="MITRE ATT&CK techniques")

plt.tight_layout()
plt.savefig("alert_dashboard.png", dpi=150)
plt.show()

## Conclusion
- Cluster aggregates endpoint, auth, and FIM logs from multiple agents.
- Custom rules map detections to MITRE ATT&CK techniques.
- Alert data can be queried and analyzed programmatically for reporting.

**Next steps:** active response (auto-block brute-force IPs), Suricata/network ingestion, vulnerability detection, threat-intel enrichment.